In [39]:
import pandas as pd

X_train = pd.read_csv("C:/Users/user/Downloads/X_train_update.csv")
y_train = pd.read_csv("C:/Users/user/Downloads/Y_train_CVw08PX.csv")

df = X_train.merge(y_train, left_index=True, right_on="Unnamed: 0")
df.head()

,Unnamed: 0,Unnamed: 0_x,designation,description,productid,imageid,Unnamed: 0_y,prdtypecode
0,0,0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,0,10
1,1,1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,1,2280
2,2,2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,2,50
3,3,3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,3,1280
4,4,4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,4,2705


In [40]:
import re
from bs4 import BeautifulSoup
import pandas as pd
from sklearn.model_selection import train_test_split

# Fonction qui nettoie un texte
def clean_text(text):

    # Si le texte est NaN (valeur manquante dans pandas)
    # on retourne une chaine vide pour éviter les erreurs
    if pd.isna(text):
        return ""

    # Supprime les balises HTML présentes dans les descriptions produits
    # Exemple : <p>Produit</p> -> Produit
    text = BeautifulSoup(text, "html.parser").get_text()

    # Convertit tout le texte en minuscules
    # Exemple : "Robot Piscine" -> "robot piscine"
    text = text.lower()

    # Supprime tous les caractères spéciaux
    # On garde seulement :
    # - lettres
    # - lettres accentuées
    # - chiffres
    # - espaces
    # Tout le reste est remplacé par un espace
    text = re.sub(r"[^a-zA-Zàâäéèêëîïôöùûüç0-9 ]", " ", text)

    # Remplace plusieurs espaces par un seul
    # Exemple : "robot     piscine" -> "robot piscine"
    text = re.sub(r"\s+", " ", text)

    # Supprime les espaces au début et à la fin de la phrase
    return text.strip()

# Création d'une nouvelle colonne "text"
# On concatène le titre du produit (designation) et la description
# fillna("") permet de remplacer les valeurs NaN par une chaine vide
df["text"] = df["designation"].fillna("") + " " + df["description"].fillna("")

# Application de la fonction de nettoyage sur toute la colonne text
# Chaque ligne est nettoyée par la fonction clean_text
df["text_clean"] = df["text"].apply(clean_text)


df["text"] = df["designation"].fillna("") + " " + df["description"].fillna("")

# nettoyage
df["text_clean"] = df["text"].apply(clean_text)

X = df["text_clean"]
y = df["prdtypecode"]

# split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

C:\Users\user\AppData\Local\Temp\ipykernel_22132\3642358058.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()
C:\Users\user\AppData\Local\Temp\ipykernel_22132\3642358058.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score,confusion_matrix
)
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

mlflow.set_tracking_uri("file:C:/Users/user/Rakuten-Challenge/mlruns")
mlflow.set_experiment("rakuten-classification")

# =========================
# PIPELINE
# =========================
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(max_iter=1000))
])


with mlflow.start_run(run_name="logreg_pipeline") as run:

    # train
    pipeline.fit(X_train, y_train)

    # predict
    preds = pipeline.predict(X_val)

    # metrics
    f1_w = f1_score(y_val, preds, average="weighted")
    f1_macro = f1_score(y_val, preds, average="macro")
    f1_micro = f1_score(y_val, preds, average="micro")

    acc = accuracy_score(y_val, preds)
    prec = precision_score(y_val, preds, average="weighted")
    rec = recall_score(y_val, preds, average="weighted")

    # log params
    mlflow.log_param("model", "logistic_regression_pipeline")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("max_features", 50000)
    mlflow.log_param("ngram_range", "(1,2)")

    # log metrics
    mlflow.log_metric("f1_weighted", f1_w)
    mlflow.log_metric("f1_macro", f1_macro)
    mlflow.log_metric("f1_micro", f1_micro)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision_weighted", prec)
    mlflow.log_metric("recall_weighted", rec)

    # =========================
    # CONFUSION MATRIX
    # =========================
    unique, counts = np.unique(y_val, return_counts=True)
    top_classes = unique[np.argsort(counts)[-20:]]

    cm = confusion_matrix(y_val, preds, labels=top_classes)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        cmap="Blues",
        xticklabels=top_classes,
        yticklabels=top_classes
    )
    plt.title("Confusion Matrix (Top 20 classes)")
    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()

    # =========================
    # TABLE EVAL
    # =========================
    eval_df = pd.DataFrame({
        "y_true": y_val,
        "y_pred": preds
    })

    mlflow.log_table(eval_df, "evaluation_table.json")

    # =========================
    # SIGNATURE (robuste)
    # =========================
    raw_examples = X_train.iloc[:5].tolist()

    input_example = []
    for v in raw_examples:
        # None / NaN -> ""
        if v is None:
            input_example.append("")
        elif isinstance(v, float) and np.isnan(v):
            input_example.append("")
        else:
            input_example.append(str(v))

    # sanity check
    assert isinstance(input_example, list)
    assert all(isinstance(x, str) for x in input_example)

    prediction_example = pipeline.predict(input_example)
    signature = infer_signature(input_example, prediction_example)


    # =========================
    # REGISTER MODEL
    # =========================
    run_id = run.info.run_id

    model_uri = f"runs:/{run_id}/model"

    mlflow.register_model(
        model_uri=model_uri,
        name="rakuten_model"
    )

    print("✅ F1 weighted :", f1_w)

2026/05/17 22:43:24 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": [
    "porte b\u00e9b\u00e9 violet et rouge trois en un m\u00e8re multifonctions kangourou fermeture \u00e0 glissi\u00e8re hoodie taille xl poitrine 104 109 cm 84 88 cm hanche 110 116 cm clair porte b\u00e9b\u00e9 violet et rouge trois en un m\u00e8re multifonctions kangourou fermeture \u00e0 glissi\u00e8re hoodie taille xl poitrine 104 109 cm 84 88 cm hanche 110 116 cm clair 1 marque nouvelle et de haute qualit\u00e9 2 d\u00e9tachable conception pratique et attentionn\u00e9e 3 parfait pour les m\u00e8res qui allaitent 4 anti vent chaud et style kangourou multifonctionnel haut de gamme 5 sac de couchage multifonction amovible de la m\u00e8re europ\u00e9enne sp\u00e9cification les typesfermezbuste104 109cmencoluresweat \u00e0 capucheles hanches110 116cmtailles disponiblesxlmat\u00e9rielcoton",
    "jesus cahiers du libre avenir pr\u00eatre autrement",
    "chambre paillasson en forme 

✅ F1 weighted : 0.8084960464341289


Registered model 'rakuten_model' already exists. Creating a new version of this model...
Created version '8' of model 'rakuten_model'.
